In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import os, re

path = "/kaggle/input/march-machine-learning-mania-2026"

# Load all data
sample   = pd.read_csv(f"{path}/SampleSubmissionStage1.csv")
mteams   = pd.read_csv(f"{path}/MTeams.csv")
wteams   = pd.read_csv(f"{path}/WTeams.csv")
mreg     = pd.read_csv(f"{path}/MRegularSeasonCompactResults.csv")
wreg     = pd.read_csv(f"{path}/WRegularSeasonCompactResults.csv")
mdet     = pd.read_csv(f"{path}/MRegularSeasonDetailedResults.csv")
wdet     = pd.read_csv(f"{path}/WRegularSeasonDetailedResults.csv")
mtourney = pd.read_csv(f"{path}/MNCAATourneyCompactResults.csv")
wtourney = pd.read_csv(f"{path}/WNCAATourneyCompactResults.csv")
massey   = pd.read_csv(f"{path}/MMasseyOrdinals.csv")
mseeds   = pd.read_csv(f"{path}/MNCAATourneySeeds.csv")
wseeds   = pd.read_csv(f"{path}/WNCAATourneySeeds.csv")
print("All files loaded. Sample rows:", len(sample))

# Seed parsing
def parse_seed(s):
    nums = re.findall(r'\d+', str(s))
    return int(nums[0]) if nums else 16

def get_seed_map(seeds_df):
    df = seeds_df.copy()
    df['SeedNum'] = df['Seed'].apply(parse_seed)
    latest = df.sort_values('Season').groupby('TeamID').last().reset_index()
    return dict(zip(latest['TeamID'], latest['SeedNum']))

mseed_map = get_seed_map(mseeds)
wseed_map = get_seed_map(wseeds)
all_seed_map = {**mseed_map, **wseed_map}

CURRENT = 2026
RECENT  = 5

# Build compact stats (win rate, margin, pts)
def build_compact_stats(teams_df, reg_df):
    recent = reg_df[reg_df['Season'] >= CURRENT - RECENT]
    rows = []
    for tid in teams_df['TeamID'].values:
        w = recent[recent['WTeamID'] == tid]
        l = recent[recent['LTeamID'] == tid]
        g = len(w) + len(l)
        if g == 0:
            rows.append([tid, 0.5, 0.0, 60.0, 60.0])
            continue
        wr  = len(w) / g
        pts = (w['WScore'].sum() + l['LScore'].sum()) / g
        pta = (w['LScore'].sum() + l['WScore'].sum()) / g
        mg  = pts - pta
        rows.append([tid, wr, mg, pts, pta])
    return pd.DataFrame(rows, columns=['TeamID','win_rate','margin','pts','pta'])

# Build detailed stats (FG%, 3P%, FT%, rebounds, assists, turnovers)
def build_detailed_stats(teams_df, det_df):
    recent = det_df[det_df['Season'] >= CURRENT - RECENT]
    rows = []
    for tid in teams_df['TeamID'].values:
        w = recent[recent['WTeamID'] == tid]
        l = recent[recent['LTeamID'] == tid]
        g = len(w) + len(l)
        if g == 0:
            rows.append([tid, 0.45, 0.33, 0.70, 35.0, 15.0, 13.0])
            continue
        fgm = (w['WFGM'].sum() + l['LFGM'].sum()) / g
        fga = (w['WFGA'].sum() + l['LFGA'].sum()) / g
        fg3m = (w['WFGM3'].sum() + l['LFGM3'].sum()) / g
        fg3a = (w['WFGA3'].sum() + l['LFGA3'].sum()) / g
        ftm = (w['WFTM'].sum() + l['LFTM'].sum()) / g
        fta = (w['WFTA'].sum() + l['LFTA'].sum()) / g
        reb = (w['WOR'].sum() + w['WDR'].sum() + l['LOR'].sum() + l['LDR'].sum()) / g
        ast = (w['WAst'].sum() + l['LAst'].sum()) / g
        tov = (w['WTO'].sum() + l['LTO'].sum()) / g
        fgpct  = fgm / fga   if fga  > 0 else 0.45
        fg3pct = fg3m / fg3a if fg3a > 0 else 0.33
        ftpct  = ftm / fta   if fta  > 0 else 0.70
        rows.append([tid, fgpct, fg3pct, ftpct, reb, ast, tov])
    return pd.DataFrame(rows, columns=['TeamID','fgpct','fg3pct','ftpct','reb','ast','tov'])

print("Building stats...")
mc = build_compact_stats(mteams, mreg)
wc = build_compact_stats(wteams, wreg)
md = build_detailed_stats(mteams, mdet)
wd = build_detailed_stats(wteams, wdet)

mstats = mc.merge(md, on='TeamID')
wstats = wc.merge(wd, on='TeamID')
all_stats = pd.concat([mstats, wstats], ignore_index=True)

# Add Massey
massey_r = massey[massey['Season'] == massey['Season'].max()]
ld = massey_r.groupby('TeamID')['RankingDayNum'].max().reset_index()
ml = massey_r.merge(ld, on=['TeamID','RankingDayNum'])
ma = ml.groupby('TeamID')['OrdinalRank'].mean().reset_index()
ma.columns = ['TeamID','massey_rank']
ma['massey_score'] = 1 - (ma['massey_rank'] / ma['massey_rank'].max())
all_stats = all_stats.merge(ma[['TeamID','massey_score']], on='TeamID', how='left')
all_stats['massey_score'] = all_stats['massey_score'].fillna(0.5)
print("Stats ready. Shape:", all_stats.shape)

# Lookup
def get_feats(tid):
    row = all_stats[all_stats['TeamID'] == tid]
    if len(row) == 0:
        return [0.5,0.0,60.0,60.0,0.45,0.33,0.70,35.0,15.0,13.0,0.5,16]
    r = row.iloc[0]
    seed = all_seed_map.get(tid, 16)
    return [r['win_rate'],r['margin'],r['pts'],r['pta'],
            r['fgpct'],r['fg3pct'],r['ftpct'],r['reb'],r['ast'],r['tov'],
            r['massey_score'], seed]

FEATS = ['wr_d','mg_d','pts_d','pta_d','fg_d','fg3_d','ft_d','reb_d','ast_d','tov_d',
         'massey_d','seed_d','t1_wr','t2_wr','t1_massey','t2_massey','t1_seed','t2_seed',
         't1_fg','t2_fg','t1_mg','t2_mg']

def make_feats(t1, t2):
    s1 = get_feats(t1)
    s2 = get_feats(t2)
    return [
        s1[0]-s2[0],   # wr diff
        s1[1]-s2[1],   # margin diff
        s1[2]-s2[2],   # pts diff
        s1[3]-s2[3],   # pta diff
        s1[4]-s2[4],   # fg% diff
        s1[5]-s2[5],   # 3p% diff
        s1[6]-s2[6],   # ft% diff
        s1[7]-s2[7],   # reb diff
        s1[8]-s2[8],   # ast diff
        s1[9]-s2[9],   # tov diff (higher = worse)
        s1[10]-s2[10], # massey diff
        s2[11]-s1[11], # seed diff (reversed: lower seed=better)
        s1[0], s2[0],  # individual wr
        s1[10], s2[10],# individual massey
        s1[11], s2[11],# individual seeds
        s1[4], s2[4],  # individual fg%
        s1[1], s2[1],  # individual margin
    ]

def build_train(df_tour):
    rows = []
    for _, row in df_tour.iterrows():
        t1 = min(row['WTeamID'], row['LTeamID'])
        t2 = max(row['WTeamID'], row['LTeamID'])
        y  = 1 if row['WTeamID'] == t1 else 0
        rows.append(make_feats(t1,t2) + [y])
    return pd.DataFrame(rows, columns=FEATS+['y'])

print("Building training data...")
train = pd.concat([build_train(mtourney), build_train(wtourney)], ignore_index=True)
X = train[FEATS].values
y = train['y'].values
print("Training rows:", len(train))

# Train XGBoost
print("Training XGBoost...")
xgb = XGBClassifier(
    n_estimators=1000, max_depth=4, learning_rate=0.02,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
    eval_metric='logloss', random_state=42, n_jobs=-1
)
xgb.fit(X, y)

# Train LightGBM
print("Training LightGBM...")
lgb = LGBMClassifier(
    n_estimators=1000, max_depth=4, learning_rate=0.02,
    subsample=0.8, colsample_bytree=0.8, min_child_samples=10,
    random_state=42, n_jobs=-1, verbose=-1
)
lgb.fit(X, y)
print("Both models trained.")

# Generate predictions
def parse_id(id_str):
    p = id_str.split('_')
    return int(p[0]), int(p[1]), int(p[2])

rows = []
for id_str in sample['ID']:
    _, t1, t2 = parse_id(id_str)
    rows.append([id_str] + make_feats(t1, t2))

test_df = pd.DataFrame(rows, columns=['ID']+FEATS)
X_test  = test_df[FEATS].values

# Ensemble: average XGBoost + LightGBM predictions
preds_xgb = xgb.predict_proba(X_test)[:, 1]
preds_lgb = lgb.predict_proba(X_test)[:, 1]
preds = (preds_xgb * 0.5) + (preds_lgb * 0.5)
preds = np.clip(preds, 0.025, 0.975)

submission = pd.DataFrame({'ID': test_df['ID'], 'Pred': preds})
submission.to_csv("/kaggle/working/submission.csv", index=False)
print("Done! Rows saved:", len(submission))
print(os.listdir("/kaggle/working"))

All files loaded. Sample rows: 519144
Building stats...
Stats ready. Shape: (760, 12)
Building training data...
Training rows: 4302
Training XGBoost...
Training LightGBM...
Both models trained.


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Done! Rows saved: 519144
['__notebook__.ipynb', 'submission.csv']


In [2]:
import glob
files = glob.glob("/kaggle/input/march-machine-learning-mania-2026/*.csv")
for f in sorted(files):
    df = pd.read_csv(f)
    print(f.split('/')[-1], '->', len(df), 'rows')

Cities.csv -> 509 rows
Conferences.csv -> 51 rows
MConferenceTourneyGames.csv -> 6793 rows
MGameCities.csv -> 90684 rows
MMasseyOrdinals.csv -> 5761702 rows
MNCAATourneyCompactResults.csv -> 2585 rows
MNCAATourneyDetailedResults.csv -> 1449 rows
MNCAATourneySeedRoundSlots.csv -> 776 rows
MNCAATourneySeeds.csv -> 2626 rows
MNCAATourneySlots.csv -> 2586 rows
MRegularSeasonCompactResults.csv -> 196823 rows
MRegularSeasonDetailedResults.csv -> 122775 rows
MSeasons.csv -> 42 rows
MSecondaryTourneyCompactResults.csv -> 1865 rows
MSecondaryTourneyTeams.csv -> 1895 rows
MTeamCoaches.csv -> 13898 rows
MTeamConferences.csv -> 13753 rows
MTeamSpellings.csv -> 1178 rows
MTeams.csv -> 381 rows
SampleSubmissionStage1.csv -> 519144 rows
SampleSubmissionStage2.csv -> 132133 rows
WConferenceTourneyGames.csv -> 6481 rows
WGameCities.csv -> 87353 rows
WNCAATourneyCompactResults.csv -> 1717 rows
WNCAATourneyDetailedResults.csv -> 961 rows
WNCAATourneySeeds.csv -> 1744 rows
WNCAATourneySlots.csv -> 1780 ro